# Neural Network Sentiment Analysis: E-Commerce Review Classification
### Framework: Discovery-to-Action (DTA) for Automated Experience Optimization
---
**Project Objective:** Build an end-to-end deep learning NLP pipeline that cleans text, builds a vocabulary mapping using an embedding architecture, trains a binary sentiment classifier, and extracts prediction confidence scores to drive automated customer service escalation logic.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, TextVectorization, Embedding, GlobalAveragePooling1D, Dense
from sklearn.model_selection import train_test_split

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")

# --- DISCOVERY PHASE: DATA INGESTION & DATASET CREATION ---
# Simulating a realistic production dataset to ensure instant execution
positive_pool = [
    "Absolutely amazing product! Highly recommend it to everyone.",
    "Excellent quality, came exactly as described and fast shipping.",
    "Very satisfied with this purchase. Works perfectly out of the box.",
    "Great customer service and fantastic item. 5 stars all the way!",
    "Exceeded my expectations, beautiful design and sturdy build quality.",
    "Love it! Will definitely be buying another one for my family.",
    "Fantastic value for money, functions exactly as it should."
] * 60

negative_pool = [
    "The product arrived broken and I am very unhappy.",
    "Terrible quality, stopped working after just two days of light use.",
    "Horrible experience, do not waste your hard earned money on this.",
    "Disappointed with the shipping time and the product feels cheap.",
    "Very poor customer support, item doesn't look like the product picture.",
    "Defective item, requested a refund but got absolutely no response.",
    "Complete waste of time, extremely poor engineering and design flaws."
] * 60

neutral_pool = [
    "It is okay, nothing special but works fine.",
    "Average item, shipping took much longer than expected.",
    "Decent for the price, but could be significantly better quality."
] * 20

text_data = positive_pool + negative_pool + neutral_pool
ratings_data = ([5] * len(positive_pool)) + ([1] * len(negative_pool)) + ([3] * len(neutral_pool))

# Build initial raw DataFrame
raw_df = pd.DataFrame({
    'review_id': [f"REV_{i:04d}" for i in range(len(text_data))],
    'review_text': text_data,
    'rating': ratings_data
})

# Artificially inject a few missing values to demonstrate data-cleaning robust practices
raw_df.loc[12, 'review_text'] = None
raw_df.loc[85, 'review_text'] = np.nan

print("\n--- Raw Data Distribution Preview ---")
print(raw_df['rating'].value_counts())
print(f"Total rows collected: {len(raw_df)} (Missing values present: {raw_df['review_text'].isna().sum()})")

In [ ]:
# --- DISCOVERY PHASE: SPECIFIC DATA CLEANING & BINARIZATION ---

# Step 1: Handle missing text values safely by dropping rows missing core content
df_clean = raw_df.dropna(subset=['review_text']).copy()

# Step 2: Remove neutral 3-star reviews to sharpen operational classification boundaries
df_clean = df_clean[df_clean['rating'] != 3].copy()

# Step 3: Convert raw star ratings to clear binary labels (4-5 stars -> 1 [Positive], 1-2 stars -> 0 [Negative])
df_clean['sentiment'] = df_clean['rating'].apply(lambda x: 1 if x >= 4 else 0)

print("--- Cleaned Label Representation ---")
print(df_clean['sentiment'].value_counts(dropna=False))
print(f"\nFinal dataset dimensions after cleaning: {df_clean.shape}")

In [ ]:
# --- DISCOVERY PHASE: STRATIFIED SPLITTING & TEXT STANDARDIZATION PLANNING ---

X = df_clean['review_text']
y = df_clean['sentiment']

# Perform a stratified split to protect label distributions across validation subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Instances: {len(X_train)} | Test/Validation Instances: {len(X_test)}")

# Configure TextVectorization parameters for production deployment
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 60

vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH,
    standardize='lower_and_strip_punctuation'
)

# Adapt the vectorizer strictly to the training data to prevent data leakage
vectorizer.adapt(X_train.values)
print("\nVocabulary mapping established successfully.")
print(f"Sample words mapped in vocabulary: {vectorizer.get_vocabulary()[:15]}")

In [ ]:
# --- TECHNICAL PHASE: MODEL ARCHITECTURE DESIGN ---

model = Sequential([
    # Input layer tracking raw unstructured string variables
    Input(shape=(), dtype=tf.string, name='raw_text_input'),

    # Text vectorization layer applying standardization and tokenization inline
    vectorizer,

    # Dense high-dimensional word embedding layer
    Embedding(input_dim=VOCAB_SIZE, output_dim=16, name='word_embedding'),

    # Dimensionality reduction layer averaging token representations
    GlobalAveragePooling1D(name='pooling_layer'),

    # Hidden representation layer
    Dense(16, activation='relu', name='dense_hidden_layer'),

    # Sigmoid output neuron mapping confidence fields between 0.0 and 1.0
    Dense(1, activation='sigmoid', name='probability_output')
])

model.summary()

In [ ]:
# --- TECHNICAL PHASE: COMPILATION & MODEL TRAINING ---

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

# Train the model over 10 epochs while monitoring convergence metrics
history = model.fit(
    tf.constant(X_train.values),
    y_train.values,
    epochs=10,
    batch_size=16,
    validation_data=(tf.constant(X_test.values), y_test.values),
    verbose=1
)

In [ ]:
# --- TECHNICAL PHASE: TRAINING METRICS VISUALIZATION ---

plt.figure(figsize=(12, 4))

# Plot accuracy trajectories
plt.subplot(1, 2, 1)
plt.plot(history.history['binary_accuracy'], label='Train Accuracy', color='#1f77b4', lw=2)
plt.plot(history.history['val_binary_accuracy'], label='Val Accuracy', color='#ff7f0e', lw=2)
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# Plot loss trajectories
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4', lw=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- ACTION PHASE: MANDATORY TARGET TEST STRING VERIFICATION ---

mandatory_test_string = "The product arrived broken and I am very unhappy"

# Format string vector safely inside a Tensor constant wrapper
prediction_tensor = tf.constant([mandatory_test_string])
raw_score = model.predict(prediction_tensor, verbose=0)[0][0]

print("=" * 65)
print(f"MANDATORY TEST STRING EVALUATION:")
print(f"Input Text: \"{mandatory_test_string}\"")
print(f"Model Sentiment Output Score (Probability): {raw_score:.4f}")
print(f"Interpreted Sentiment Classification: {'POSITIVE' if raw_score >= 0.5 else 'NEGATIVE'}")
print("=" * 65)

In [ ]:
# --- ACTION PHASE: RUNTIME AUTOMATED SUPPORT ROUTING SIMULATOR ---

def automated_support_router(review_text, model_instance, low_threshold=0.20, high_threshold=0.80):
    """
    Evaluates raw review strings and routes them dynamically based on confidence thresholds.
    """
    score = model_instance.predict(tf.constant([review_text]), verbose=0)[0][0]

    if score <= low_threshold:
        action = "⚠️ CRITICAL ESCALATION: Auto-routed instantly to high-priority customer support queue."
        confidence = (1.0 - score) * 100
    elif score >= high_threshold:
        action = "✅ AUTO-ARCHIVE: Verified positive review. Sent to marketing showcase pipeline."
        confidence = score * 100
    else:
        action = "🔍 HUMAN TRIAGE QUEUE: Mid-tier confidence score. Scheduled for manual oversight review."
        confidence = max(score, 1.0 - score) * 100

    return {
        'text': review_text,
        'probability_score': round(float(score), 4),
        'operational_action': action,
        'confidence_percentage': round(confidence, 2)
    }

# Simulating real-world streams to evaluate the decision logic
test_stream = [
    "The product arrived broken and I am very unhappy",
    "It works okay, nothing great, nothing bad.",
    "This is the most incredible tool I have used all year!"
]

print("\n--- Live Production Support Routing Dashboard Simulation ---")
for text in test_stream:
    route_result = automated_support_router(text, model)
    print(f"\nReview: '{route_result['text']}'")
    print(f" -> Output Score: {route_result['probability_score']}")
    print(f" -> Confidence: {route_result['confidence_percentage']}%")
    print(f" -> Action Taken: {route_result['operational_action']}")